# 22a. Chuẩn bị demo: cắt 3 đoạn D1, D2, D3 và chạy bộ phát hiện trên từng đoạn (GĐ J, task 8.2), CPU
Cắt 3 đoạn demo (mục 3 của `report/KICH_BAN_DEMO.md`) từ `NF-ToN-IoT-v3.csv` thành file nhỏ (giữ đủ flow, không lấy mẫu); kiểm tra phần trước mốc tấn công
chỉ có lưu lượng bình thường (nếu không thì dời mốc bắt đầu và ghi lại); chạy `scripts/detector.py` trên từng đoạn với hybrid chính và hybrid nhẹ
(có đối chiếu đồ thị offline để biết nhãn thật); so thời gian phát hiện với kịch bản; xuất `demo_replay.json` cho trang `report/demo/demo_replay.html`
và 3 file đoạn cho notebook `22_demo`. Không sửa mô hình, cấu hình, ngưỡng đã đóng băng.

In [ ]:
import glob, json, math, os, resource, subprocess, sys, time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

SEED = 42
WORK = "/kaggle/working"
TMP = "/tmp/prep"
REPO = "/tmp/gnn-iot-ids"
URL = "https://github.com/minucedev/gnn-iot-ids.git"
INFO = {"steps_s": {}, "device": "CPU", "gpu": None, "seed": SEED}
T_START = time.time()


def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return "không rõ"


def peak_ram_gb():
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def jdefault(o):
    """Chuyển số numpy sang số Python khi ghi JSON."""
    return o.item() if hasattr(o, "item") else str(o)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    INFO["gpu_hours"] = 0
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1, default=jdefault)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), log=None, **kw):
    if log:
        with open(log, "w", encoding="utf-8") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, text=True, **kw)
        print(open(log, encoding="utf-8").read()[-6000:])
    else:
        r = subprocess.run(cmd, capture_output=True, text=True, **kw)
        print(r.stdout[-9000:], end="")
        print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Không thấy {pattern} trong /kaggle/input (đã gắn đủ nguồn dữ liệu chưa?)"
    return hits[0]


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", URL, REPO])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])
INFO["cpu"], INFO["cpu_count"] = cpu_model(), os.cpu_count()
save_info()
print("Commit:", INFO["commit"], "| CPU:", INFO["cpu"])

## pytest (dừng nếu lỗi): kiểm chạy mặc định của detector vẫn y hệt, các tùy chọn mới, đóng gói demo

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q", "tests/test_detector_stream.py", "tests/test_demo_replay.py", "tests/test_stream_features.py",
             "tests/test_realtime_eval.py", "tests/test_build_graph.py"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]
save_info()

## Đường dẫn đầu vào (như notebook 20)

In [ ]:
DS = "nf_ton_iot_v3"
OLD = f"{REPO}/reports/kaggle"
SPEC = {DS: dict(data=os.path.dirname(find(f"{DS}/split.json")),
                 ckpt=os.path.dirname(find(f"ckpt/{DS}_gat_s42.pt")),
                 folds=os.path.dirname(os.path.dirname(find(f"models_hybrid/{DS}_s42/lgbm_oofavg_il.pkl"))),
                 light=os.path.dirname(os.path.dirname(find(f"models_light/{DS}_s42/lgbm_n100_l15.pkl"))),
                 baseline_search=f"{OLD}/gnn-iot-ids-05-baselines/baseline_search_{DS}.json",
                 ablation_search=f"{OLD}/gnn-iot-ids-16d-ablation-ton/ablation_search_{DS}.json",
                 gnn_csv=f"{OLD}/gnn-iot-ids-07-train-gnn/results_gnn_{DS}.csv",
                 hybrid_csv=f"{OLD}/gnn-iot-ids-15f-hybrid-il-ton/results_hybrid_oofavg_il_{DS}.csv")}
CFG = dict(dataset=DS, split="test", flows_csv=find("NF-ToN-IoT-v3.csv"), expect_commit=INFO["commit"][:7],
           thresholds=f"{OLD}/gnn-iot-ids-16c-aggregate/thresholds.json",
           thresholds_c={DS: f"{OLD}/gnn-iot-ids-16h-c-threshold/thresholds_c_{DS}.json"}, datasets=SPEC)
for k, v in SPEC[DS].items():
    assert os.path.exists(v), f"{k}: không thấy {v}"
for p in (CFG["thresholds"], *CFG["thresholds_c"].values(), CFG["flows_csv"]):
    assert os.path.exists(p), p
CFG_PATH = f"{WORK}/detector_config.json"
json.dump(CFG, open(CFG_PATH, "w"), ensure_ascii=False, indent=1)
AHP = json.load(open(f"{REPO}/reports/tables/ahp_result.json", encoding="utf-8"))
MAIN = AHP["recommended"]
LIGHT = "hybrid_light"
MODELS = [MAIN, LIGHT]
INFO["models"] = MODELS
save_info()
print("Mô hình chính (AHP gợi ý):", MAIN, "| nhẹ:", LIGHT, "| CSV:", CFG["flows_csv"], f"{os.path.getsize(CFG['flows_csv']) / 2**30:.2f} GB")

## Cắt 3 đoạn từ CSV gốc
Đọc CSV theo từng khối (mọi cột là chuỗi, giữ nguyên giá trị), giữ mọi flow có `FLOW_START_MILLISECONDS` trong khoảng (lấy rộng thêm 10 phút phía trước để
còn dời mốc bắt đầu nếu cần). Không lấy mẫu.

In [ ]:
import numpy as np
import pandas as pd

sys.path.insert(0, REPO)
from src import build_graph as bg  # noqa: E402
from src import demo_replay as DR  # noqa: E402


def epoch(s):
    return datetime.fromisoformat(s).replace(tzinfo=timezone.utc).timestamp()


def iso(t):
    return datetime.fromtimestamp(float(t), tz=timezone.utc).isoformat(timespec="seconds")


PLAN = {
    "D1": dict(title="Yên tĩnh rồi DDoS", cls="DoS_DDoS", start="2019-04-24T20:07:00", end="2019-04-24T20:12:00", onset="2019-04-24T20:09:15", doc_s=5.0),
    "D2": dict(title="Dò quét", cls="Truy_cap_trai_phep", start="2019-04-26T18:29:30", end="2019-04-26T18:33:00", onset="2019-04-26T18:30:45", doc_s=5.1),
    "D3": dict(title="Backdoor", cls="Botnet_Backdoor", start="2019-04-28T15:27:00", end="2019-04-28T15:31:00", onset="2019-04-28T15:28:35", doc_s=4.4),
}
MARGIN_S, CHUNK = 600, 250_000
TS_COL, LAB_COL = "FLOW_START_MILLISECONDS", "Attack"
lo = {k: (epoch(v["start"]) - MARGIN_S) * 1000 for k, v in PLAN.items()}
hi = {k: epoch(v["end"]) * 1000 for k, v in PLAN.items()}
parts = {k: [] for k in PLAN}
n_read, tcol, lcol = 0, None, None
with step("doc_csv_cat_doan"):
    for chunk in pd.read_csv(CFG["flows_csv"], dtype=str, keep_default_na=False, na_filter=False, chunksize=CHUNK, encoding="utf-8-sig"):
        if tcol is None:
            names = {c.strip(): c for c in chunk.columns}
            tcol, lcol = names[TS_COL], names[LAB_COL]
        n_read += len(chunk)
        t = pd.to_numeric(chunk[tcol], errors="coerce")
        for k in PLAN:
            m = ((t >= lo[k]) & (t < hi[k])).to_numpy()
            if m.any():
                parts[k].append(chunk[m])
RAW = {k: pd.concat(v, ignore_index=True) for k, v in parts.items()}
INFO["csv_rows_read"] = n_read
print(f"Đã đọc {n_read:,} dòng CSV; flow trong khoảng mở rộng:", {k: len(v) for k, v in RAW.items()})

## Mốc tấn công thật, kiểm tra phần yên tĩnh trước mốc, dời mốc bắt đầu nếu cần, ghi file đoạn

In [ ]:
DATA = f"{WORK}/demo_data"
os.makedirs(DATA, exist_ok=True)
SEG = {}
for k, p in PLAN.items():
    raw = RAW[k]
    tm = pd.to_numeric(raw[tcol]).to_numpy(np.int64)
    t = tm / 1000.0
    lab = np.asarray(bg.to_group(raw[lcol]))
    nominal = epoch(p["onset"])
    cand = t[(lab == p["cls"]) & (t >= nominal - 1.0)]
    assert len(cand), f"{k}: không có flow {p['cls']} từ mốc {p['onset']}"
    onset = float(cand.min())
    assert 0 <= onset - nominal < 2.0, f"{k}: mốc thật {iso(onset)} lệch mốc kịch bản {p['onset']}"
    start0, end = epoch(p["start"]), epoch(p["end"])
    pre = (t >= start0) & (t < onset)
    bad = pre & (lab != "Normal")
    start = start0
    moved = None
    if bad.any():                                           # có lưu lượng tấn công/khác trước mốc: dời mốc bắt đầu tới cửa sổ 5 s ngay sau flow đó
        last_bad = float(t[bad].max())
        start = math.ceil((last_bad + 1e-3) / 5) * 5
        moved = {"from": p["start"], "to": iso(start), "n_non_normal_before": int(bad.sum()), "labels": pd.Series(lab[bad]).value_counts().to_dict(),
                 "last_non_normal": iso(last_bad)}
    assert onset - start >= 30, f"{k}: sau khi dời mốc chỉ còn {onset - start:.0f} s yên tĩnh trước mốc tấn công"
    sel = (tm >= start * 1000) & (tm < end * 1000)
    path = f"{DATA}/{k}.csv"
    raw[sel].to_csv(path, index=False, encoding="utf-8")
    comp_pre = pd.Series(lab[(t >= start) & (t < onset)]).value_counts().to_dict()
    comp_post = pd.Series(lab[(t >= onset) & (t < end)]).value_counts().to_dict()
    SEG[k] = dict(id=k, title=p["title"], cls=p["cls"], start=iso(start), start_planned=p["start"], end=p["end"], onset_nominal=p["onset"], onset_ts=onset,
                  onset=datetime.fromtimestamp(onset, tz=timezone.utc).isoformat(timespec="milliseconds"), onset_window=iso(math.floor(onset / 5) * 5),
                  start_moved=moved, n_flows=int(sel.sum()), file=path, doc_s=p["doc_s"], flows_before_onset=comp_pre, flows_after_onset=comp_post,
                  duration_s=end - start, calm_s=onset - start)
    print(f"{k} {p['title']}: {iso(start)} → {p['end']}, mốc thật {SEG[k]['onset']}, {int(sel.sum()):,} flow ({os.path.getsize(path) / 1024:.0f} KB)")
    print(f"    trước mốc: {comp_pre} | từ mốc: {comp_post}" + (f" | ĐÃ DỜI MỐC BẮT ĐẦU: {moved}" if moved else " | phần trước mốc chỉ có Normal, không dời"))
INFO["segments"] = SEG
json.dump(SEG, open(f"{WORK}/segments.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=jdefault)
save_info()
del RAW, parts

## Kiểm tra file đoạn: làm sạch bằng đúng hàm offline, nhãn trước mốc chỉ có Normal, mọi cửa sổ thuộc tập test

In [ ]:
meta = json.load(open(os.path.join(SPEC[DS]["data"], "split.json"), encoding="utf-8"))
test_wins = set(int(w) for w in meta["test"]["windows"])
for k, s in SEG.items():
    flows, classes, log = bg.load_clean(DS, s["file"])
    wins = np.unique(np.floor(flows["ts"].to_numpy() / 5).astype(np.int64))
    outside = [int(w) for w in wins if int(w) not in test_wins]
    pre = flows[flows["ts"] < s["onset_ts"]]
    s["check"] = {"n_flows_clean": int(len(flows)), "n_windows": int(len(wins)), "classes": list(classes), "windows_outside_test": len(outside),
                  "labels_before_onset": sorted(set(pre["label"])), "n_before_onset": int(len(pre)), "dropped_by_clean": int(s["n_flows"] - len(flows))}
    print(k, s["check"])
    assert not outside, f"{k}: {len(outside)} cửa sổ không thuộc tập test"
    assert set(pre["label"]) == {"Normal"}, f"{k}: trước mốc còn nhãn {set(pre['label'])}"
    assert s["check"]["dropped_by_clean"] <= 0.01 * s["n_flows"], f"{k}: làm sạch bỏ quá nhiều flow"
save_info()

## Chạy bộ phát hiện trên từng đoạn, hybrid chính và hybrid nhẹ
Nạp mỗi mô hình một lần (`detector.prepare`, có đối chiếu đồ thị offline để biết nhãn thật), rồi chạy từng đoạn bằng `detector.main` với các tùy chọn mới
(`--start`, `--end`, `--flows-csv`, `--partial-flows`, `--alert-from`, `--dump-first-alert`), không chờ thời gian thật. Thời gian phát hiện tính như
`scripts/realtime_eval.py` (cùng hàm `episodes`).

In [ ]:
import importlib.util


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


DET = load_module("detector", f"{REPO}/scripts/detector.py")
RE = load_module("realtime_eval", f"{REPO}/scripts/realtime_eval.py")
CFGD = json.load(open(CFG_PATH, encoding="utf-8"))
SESS = {}
for m in MODELS:
    with step(f"nap_{m}"):
        SESS[m] = DET.prepare(CFGD, m, SEED, reference=True)

RUNS = {}
for k, s in SEG.items():
    for m in MODELS:
        out = f"{TMP}/{k}_{m}"
        os.makedirs(out, exist_ok=True)
        with step(f"chay_{k}_{m}"):
            DET.main(["--config", CFG_PATH, "--model", m, "--seed", str(SEED), "--out", out, "--flows-csv", s["file"], "--partial-flows",
                      "--start", s["start"], "--end", s["end"], "--alert-from", s["onset_window"], "--dump-first-alert", f"{out}/first_alert.json"], SESS[m])
        win = pd.read_csv(f"{out}/windows.csv")
        alerts = pd.read_csv(f"{out}/alerts.csv")
        k_cls = SESS[m].classes.index(s["cls"])
        ep = DR.pick_episode(RE.episodes(win, k_cls, 60.0, 5), s["onset_ts"])
        w0 = math.floor(s["onset_ts"] / 5) * 5
        calm = win[(win["t_start"] < w0) & (win["n_alert"] > 0)]
        after = win[(win["t_start"] >= w0) & (win["n_alert"] > 0)]
        ref_ok = win["ref_found"] == 1
        RUNS[(k, m)] = dict(out=out, win=win, alerts=alerts, ep=ep,
                            calm_alerts=[{"t": DR.hms(r.t_close), "n_alert": int(r.n_alert)} for r in calm.itertuples()],
                            first_alert_t=None if after.empty else DR.hms(after.iloc[0]["t_close"]),
                            agreement=dict(n_windows=int(len(win)), n_ref=int(ref_ok.sum()), n_mis_thr=int(win["n_mis_thr"].sum()), n_mis_arg=int(win["n_mis_arg"].sum()),
                                           struct_bad=int((win["struct_ok"] == 0).sum()), max_dx=float(win["max_dx"].max()), max_de=float(win["max_de"].max()),
                                           max_dproba=float(win["max_dproba"].max())),
                            p95_ms=float(np.percentile(win["total_ms"], 95)))
save_info()

## Kết quả từng đoạn, so với kịch bản

In [ ]:
rows = []
for k, s in SEG.items():
    for m in MODELS:
        R = RUNS[(k, m)]
        ep = R["ep"]
        rows.append({"đoạn": k, "mô hình": m, "mốc tấn công thật": s["onset"][11:23], "phát hiện": ep["detected"], "cảnh báo đầu tiên từ mốc": R["first_alert_t"],
                     "thời gian phát hiện (s)": None if ep["detection_s"] is None else round(ep["detection_s"], 3), "kịch bản (s)": s["doc_s"],
                     "báo nhầm trước mốc (cửa sổ)": len(R["calm_alerts"]), "p95 ms/cửa sổ": round(R["p95_ms"], 1),
                     "lệch offline (node)": R["agreement"]["n_mis_thr"], "cửa sổ lệch cấu trúc": R["agreement"]["struct_bad"]})
TAB = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 30)
print(TAB.to_string(index=False))
for k, s in SEG.items():
    s["detection"] = {m: {"detected": RUNS[(k, m)]["ep"]["detected"], "detection_s": RUNS[(k, m)]["ep"]["detection_s"],
                          "calm_alerts": RUNS[(k, m)]["calm_alerts"], "first_alert_t": RUNS[(k, m)]["first_alert_t"],
                          "agreement": RUNS[(k, m)]["agreement"], "p95_ms": RUNS[(k, m)]["p95_ms"]} for m in MODELS}
    for m in MODELS:
        d = RUNS[(k, m)]["ep"]["detection_s"]
        if d is not None and abs(d - s["doc_s"]) > 0.05:
            print(f"LỆCH KỊCH BẢN: {k} {m}: đo được {d:.3f} s, kịch bản ghi {s['doc_s']} s")
INFO["result_table"] = TAB.to_dict("records")
save_info()

## Thử dòng lệnh thật (python -I scripts/detector.py ...): in từng cửa sổ, tô đỏ, dừng sau cảnh báo đầu tiên từ mốc tấn công

In [ ]:
s = SEG["D2"]
with step("cli_smoke"):
    r = run([sys.executable, "-I", "scripts/detector.py", "--config", CFG_PATH, "--model", LIGHT, "--out", f"{TMP}/cli_smoke", "--flows-csv", s["file"],
             "--partial-flows", "--no-reference", "--start", s["start"], "--end", s["end"], "--speedup", "60", "--print-windows", "--color",
             "--stop-after-alert", "--alert-from", s["onset_window"], "--dump-first-alert", f"{TMP}/cli_smoke/first_alert.json"], cwd=REPO)
assert "] cửa sổ" in r.stdout and "⚠" in r.stdout and os.path.exists(f"{TMP}/cli_smoke/first_alert.json")
INFO["cli_smoke_tail"] = r.stdout.strip().splitlines()[-6:]
save_info()

## Xuất demo_replay.json (cho trang HTML) và file đoạn (cho notebook 22)

In [ ]:
speed = DR.speed_numbers(REPO)
replay = {
    "meta": {"commit": INFO["commit"], "generated_utc": iso(time.time()), "dataset": DS, "split": "test", "window_sec": 5, "speedup_default": 5,
             "main_model": MAIN, "light_model": LIGHT, "seed": SEED, "cpu": INFO["cpu"], "threads": 1, "alert_definition": "p = 1 - P(Normal), vượt ngưỡng đã đóng băng",
             "models": {m: {"threshold": SESS[m].model.threshold, "n_params": SESS[m].model.n_params, "size_kb": SESS[m].model.size_kb} for m in MODELS}},
    "segments": [],
    "speed": {"numbers": speed, "rows": DR.speed_rows(speed)},
    "limits": DR.limits_rows(REPO),
}
for k, s in SEG.items():
    runs = {}
    for m in MODELS:
        R = RUNS[(k, m)]
        fa = json.load(open(f"{R['out']}/first_alert.json", encoding="utf-8"))
        assert fa is not None, f"{k} {m}: không có cảnh báo nào từ mốc tấn công"
        runs[m] = {"windows": DR.pack_windows(R["win"], R["alerts"]), "first_alert_graph": DR.shrink_graph(fa),
                   "detected": R["ep"]["detected"], "detection_s": R["ep"]["detection_s"], "first_alert_t": R["first_alert_t"], "calm_alerts": R["calm_alerts"]}
    replay["segments"].append({"id": k, "title": s["title"], "cls": s["cls"], "start": s["start"], "end": s["end"], "onset": s["onset"], "onset_t": s["onset"][11:19],
                               "n_flows": s["n_flows"], "start_moved": s["start_moved"], "runs": runs})
with open(f"{WORK}/demo_replay.json", "w", encoding="utf-8") as f:
    json.dump(replay, f, ensure_ascii=False, separators=(",", ":"), default=jdefault)
INFO["replay_json_kb"] = round(os.path.getsize(f"{WORK}/demo_replay.json") / 1024, 1)
TAB.to_csv(f"{WORK}/segments_result.csv", index=False)
save_info()
print("demo_replay.json:", INFO["replay_json_kb"], "KB;", {s["id"]: {m: len(s["runs"][m]["windows"]) for m in MODELS} for s in replay["segments"]})
print("Đồ thị cửa sổ cảnh báo đầu tiên:", {s["id"]: {m: (s["runs"][m]["first_alert_graph"]["t"], s["runs"][m]["first_alert_graph"]["n_nodes_total"],
                                                        s["runs"][m]["first_alert_graph"]["n_edges_total"], s["runs"][m]["first_alert_graph"]["n_alert_total"]) for m in MODELS} for s in replay["segments"]})
print(os.listdir(WORK), os.listdir(DATA))